# Rx PD Model - Simplified Colab Version
## No dependency conflicts - uses only core libraries

In [ ]:
# Minimal install - only essential libraries
!pip install -q --upgrade pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, precision_score, recall_score, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print('✓ All libraries imported!')

# Step 1: Extract and Load Data

In [ ]:
# List files
print('Files in directory:')
for f in os.listdir('.'):
    if f.endswith(('.zip', '.csv')):
        print(f'  {f}')

In [ ]:
# Extract zips
for zip_file in ['Lending_default_train.zip', 'Lending_default_holdout.zip']:
    if os.path.exists(zip_file):
        print(f'Extracting {zip_file}...')
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall()
        print('  ✓ Done')

In [ ]:
# Load data
print('Loading data...')
df_train_tx = pd.read_csv('Lending_default_train_tx.csv')
df_train_acc = pd.read_csv('Lending_default_train_account.csv')
df_train_label = pd.read_csv('Lending_default_train_label.csv')
df_holdout_tx = pd.read_csv('Lending_default_holdout_tx.csv')
df_holdout_acc = pd.read_csv('Lending_default_holdout_account.csv')

# Clean columns
for df in [df_train_tx, df_train_acc, df_train_label, df_holdout_tx, df_holdout_acc]:
    cols = [c for c in df.columns if c.startswith('Unnamed')]
    df.drop(columns=cols, inplace=True)

print('✓ Data loaded!')

# Step 2: Merge Datasets

In [ ]:
# Merge training
df_train = df_train_tx.merge(df_train_acc, on='Restaurant_ID').merge(df_train_label, on='Restaurant_ID')

# Merge holdout
df_holdout = df_holdout_tx.merge(df_holdout_acc, on='Restaurant_ID')

print(f'Training: {df_train.shape}')
print(f'Holdout: {df_holdout.shape}')
print(f'\nEvent rate: {df_train["loan_default"].mean()*100:.2f}%')

# Step 3: 80/20 Train-Test Split

In [ ]:
# Prepare
y = df_train['loan_default']
X = df_train.drop(columns=['loan_default', 'Restaurant_ID', 'Tx_date'])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Create dfs
df_train_set = X_train.copy()
df_train_set['loan_default'] = y_train.values
df_test_set = X_test.copy()
df_test_set['loan_default'] = y_test.values

print('='*70)
print('TRAIN-TEST SPLIT')
print('='*70)
print(f'Training: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)')
print(f'\nEvent rates:')
print(f'  Train: {y_train.mean()*100:.2f}%')
print(f'  Test: {y_test.mean()*100:.2f}%')

# Step 4: Train Models

In [ ]:
# Logistic Regression
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)

df_train_set['pred_lr'] = lr.predict_proba(X_train)[:, 1]
df_test_set['pred_lr'] = lr.predict_proba(X_test)[:, 1]

auc_train_lr = roc_auc_score(y_train, df_train_set['pred_lr'])
auc_test_lr = roc_auc_score(y_test, df_test_set['pred_lr'])
print(f'  Train AUC: {auc_train_lr:.4f}')
print(f'  Test AUC: {auc_test_lr:.4f}')

In [ ]:
# Gradient Boosting
print('Training Gradient Boosting...')
gb = GradientBoostingClassifier(
    n_estimators=100, max_leaf_nodes=10, learning_rate=0.1, 
    min_samples_leaf=0.05, random_state=42
)
gb.fit(X_train, y_train)

df_train_set['pred_gb'] = gb.predict_proba(X_train)[:, 1]
df_test_set['pred_gb'] = gb.predict_proba(X_test)[:, 1]

auc_train_gb = roc_auc_score(y_train, df_train_set['pred_gb'])
auc_test_gb = roc_auc_score(y_test, df_test_set['pred_gb'])
print(f'  Train AUC: {auc_train_gb:.4f}')
print(f'  Test AUC: {auc_test_gb:.4f}')

# Step 5: Model Comparison

In [ ]:
print('='*70)
print('MODEL COMPARISON')
print('='*70)
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Gradient Boosting'],
    'Train AUC': [auc_train_lr, auc_train_gb],
    'Test AUC': [auc_test_lr, auc_test_gb]
})
print(comparison.round(4).to_string(index=False))

best_idx = 0 if auc_test_lr > auc_test_gb else 1
best_name = ['Logistic Regression', 'Gradient Boosting'][best_idx]
best_model = [lr, gb][best_idx]
best_pred_col = ['pred_lr', 'pred_gb'][best_idx]
best_auc = max(auc_test_lr, auc_test_gb)

print(f'\n✓ Best: {best_name} (AUC: {best_auc:.4f})')

# Step 6: Performance Metrics

In [ ]:
def show_metrics(y_true, y_pred_prob, name='Test'):
    y_pred = (y_pred_prob > 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    
    print(f'\n{name} Set Metrics:')
    print(f'  Accuracy: {accuracy_score(y_true, y_pred):.4f}')
    print(f'  Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}')
    print(f'  Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'  Recall: {recall_score(y_true, y_pred):.4f}')
    print(f'  ROC-AUC: {roc_auc_score(y_true, y_pred_prob):.4f}')
    print(f'  Confusion Matrix:')
    print(f'    TN: {cm[0,0]:,}  FP: {cm[0,1]:,}')
    print(f'    FN: {cm[1,0]:,}  TP: {cm[1,1]:,}')

print('='*70)
print(f'{best_name.upper()} PERFORMANCE')
print('='*70)
show_metrics(y_train, df_train_set[best_pred_col], 'Train')
show_metrics(y_test, df_test_set[best_pred_col], 'Test')

# Step 7: Time Series - Actual vs Predicted by Period

In [ ]:
def plot_calibration(df_data, pred_col, actual_col, title='Test', n_periods=10):
    df_temp = df_data.copy()
    df_temp['period'] = pd.qcut(range(len(df_temp)), q=n_periods, 
                                 labels=[f'P{i+1}' for i in range(n_periods)])
    
    ts = df_temp.groupby('period').agg({
        actual_col: 'mean',
        pred_col: 'mean'
    })
    
    ts['actual_pct'] = ts[actual_col] * 100
    ts['pred_pct'] = ts[pred_col] * 100
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    x = range(len(ts))
    
    ax.plot(x, ts['actual_pct'], marker='o', linewidth=2.5, markersize=8,
            label='Actual Default Rate', color='#d62728')
    ax.plot(x, ts['pred_pct'], marker='s', linewidth=2.5, markersize=8,
            label='Predicted Default Rate', color='#1f77b4')
    
    ax.set_xlabel('Period', fontsize=12, fontweight='bold')
    ax.set_ylabel('Default Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'{title}: Actual vs Predicted Default Rates', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ts.index)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Metrics
    mae = abs(ts['actual_pct'] - ts['pred_pct']).mean()
    print(f'{title} Calibration: MAE = {mae:.2f}%')
    print(ts[['actual_pct', 'pred_pct']].round(2).to_string())

print('='*70)
print('TRAINING DATA: Actual vs Predicted')
print('='*70)
plot_calibration(df_train_set, best_pred_col, 'loan_default', 'Training')

In [ ]:
print('\n' + '='*70)
print('TEST DATA: Actual vs Predicted')
print('='*70)
plot_calibration(df_test_set, best_pred_col, 'loan_default', 'Test')

# Step 8: Score Holdout Data

In [ ]:
# Prepare holdout
X_holdout = df_holdout.drop(columns=[c for c in df_holdout.columns if c in ['Restaurant_ID', 'Tx_date']])

# Align features
for col in X_train.columns:
    if col not in X_holdout.columns:
        X_holdout[col] = 0
X_holdout = X_holdout[X_train.columns]

# Score
holdout_pred = best_model.predict_proba(X_holdout)[:, 1]

# Results
results = pd.DataFrame({
    'Restaurant_ID': df_holdout['Restaurant_ID'].values,
    'Predicted_Default_Probability': np.round(holdout_pred, 4),
    'Predicted_Default_Score_0_100': np.round(holdout_pred * 100, 2)
})

# Save
results.to_csv('holdout_predictions.csv', index=False)

print('='*70)
print(f'HOLDOUT SCORING - {best_name}')
print('='*70)
print(f'Total: {len(results):,} records')
print(f'\nPrediction Distribution:')
print(results['Predicted_Default_Probability'].describe().round(4))
print(f'\nFirst 10:')
print(results.head(10).to_string(index=False))
print(f'\n✓ Saved to holdout_predictions.csv')

# Step 9: Holdout Distribution

In [ ]:
# Distribution plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(results['Predicted_Default_Probability'], bins=50, color='#1f77b4', alpha=0.7, edgecolor='black')
ax.axvline(results['Predicted_Default_Probability'].mean(), color='#d62728', linestyle='--', linewidth=2, label='Mean')
ax.set_xlabel('Predicted Default Probability', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Holdout: Prediction Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mean prediction: {results["Predicted_Default_Probability"].mean():.4f}')
print(f'Std dev: {results["Predicted_Default_Probability"].std():.4f}')
print(f'Min: {results["Predicted_Default_Probability"].min():.4f}')
print(f'Max: {results["Predicted_Default_Probability"].max():.4f}')

# Summary

In [ ]:
print('='*70)
print('FINAL SUMMARY')
print('='*70)
print(f'\nData:')
print(f'  Training: {len(X_train):,}')
print(f'  Test: {len(X_test):,}')
print(f'  Holdout: {len(results):,}')
print(f'\nBest Model: {best_name}')
print(f'  Test AUC: {best_auc:.4f}')
print(f'\nOutput:')
print(f'  ✓ holdout_predictions.csv')
print(f'\nTo download in Colab:')
print(f'  1. Click folder icon on left')
print(f'  2. Right-click holdout_predictions.csv')
print(f'  3. Select Download')
print(f'\n✓ Complete!')